# 09 - Spark Connect

Run Spark from this notebook against the in-stack **Spark Connect** sidecar.
There is no Spark distribution in this image — `pyspark-client` is a thin gRPC
client (~1.5 MB, no JVM). The driver runs on the `spark-connect` container,
executors on `spark-worker`, and results stream back here.

`SPARK_REMOTE` is injected by Compose (default `sc://spark-connect:15002`).
For the in-stack sidecar, enable Spark with `./start.sh --spark-source container` (`SPARK_SOURCE != disabled`).

Override `SPARK_REMOTE` for a remote Spark Connect server using Spark 4.1.
The kernel uses `pyspark-client==4.1.2`.

EMR Serverless requires a separate Python environment that matches its Spark version.
Use `services/spark/examples/emr_serverless_connect.py` with the version pins in that file.

In [ ]:
import os
from pyspark.sql import SparkSession
from urllib.parse import urlsplit
from ipaddress import IPv6Address
import re

def _spark_endpoint_display(address):
    """Keep connection options and user credentials out of notebook output."""
    try:
        endpoint = urlsplit(address)
        host = endpoint.hostname
        if not host or endpoint.scheme != "sc" or "%" in host:
            return "<configured endpoint>"
        if ":" in host:
            host = f"[{IPv6Address(host)}]"
        elif re.fullmatch(r"[A-Za-z0-9.-]+", host) is None:
            return "<configured endpoint>"
        port = "" if endpoint.port is None else f":{endpoint.port}"
        return f"{endpoint.scheme}://{host}{port}"
    except ValueError:
        return "<configured endpoint>"


spark_remote = os.environ.get("SPARK_REMOTE", "sc://spark-connect:15002").strip()
spark_remote_display = _spark_endpoint_display(spark_remote)

try:
    spark = SparkSession.builder.remote(spark_remote).getOrCreate()
    print("connected:", spark_remote_display, "| Spark", spark.version)
except Exception as exc:  # noqa: BLE001
    raise RuntimeError(
        f"Could not reach Spark Connect at {spark_remote_display!r} ({type(exc).__name__}).\n"
        "Is Spark enabled? Re-run ./start.sh --spark-source container (the "
        "spark-connect sidecar's JVM also lags spark-master by 20-60s on a "
        "cold start). The client's pyspark-client version must match the "
        "server's Spark version (4.1.2)."
    ) from None

## DataFrame + SQL
These run on the remote driver, not in this kernel.

In [ ]:
from pyspark.sql.functions import col

df = spark.range(10).withColumn("squared", col("id") * col("id"))
df.show()
print("count:", df.count())
spark.sql("SELECT 1 + 1 AS result").show()

## MinIO (s3a) round-trip
Storage credentials live on the **server** (the sidecar's conf), so `s3a://`
just works from here — no keys in the notebook. Writes also land in the Spark
History Server automatically (`spark.eventLog` is set server-side).

In [ ]:
path = f"s3a://{os.environ.get('MINIO_BUCKET_SPARK_HISTORY', 'spark-history')}/_notebook09_probe"
spark.range(5).write.mode("overwrite").parquet(path)
print("rows read back:", spark.read.parquet(path).count())

In [ ]:
spark.stop()